## c23 - Design Space Optimizer
CMA-ES that jointly minimises LCA cost and maximises reclaimed-timber reuse. Each candidate passes through: geometry → feasibility → cost matrix → MILP → GNN → fitness. Run sections in order: 1 Setup → 2 Evaluation → 3 GA run → 4 Analysis → 5 Export.

# 1. Setup
Load all dependencies, stock data, and search space. Configure GA weights, MILP constraints, and the surrogate model prefix. **Run once per session** before the evaluation or GA cells.

In [ ]:
import importlib
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import config
import c00_headquarter_params as c11_params
from workflows import c22_stage_geometry             as stage_geometry
from workflows import c24_stage_feasibility          as stage_feas
from workflows import c25_stage_cost_matrix          as stage_cost
from workflows import c26_stage_MILP                 as stage_milp
from workflows import c27_stage_GNN                  as stage_gnn
from workflows import c28_stage_fitness_score        as stage_fitness
from workflows import c28_stage_normalization_bounds as stage_bounds
from workflows import c23_ga_evaluator               as ga_eval

for _mod in [stage_geometry, stage_feas, stage_cost,
             stage_milp, stage_gnn, stage_fitness, stage_bounds]:
    importlib.reload(_mod)
importlib.reload(ga_eval)

SEED = 42
random.seed(SEED)

TESTING = False
TRAINING_SCENARIO = "A"  # "new", "A", "B"

json_path = config.DATA_IO_PATH / f"search_space_{c11_params.GRID}.json"
if not json_path.exists():
    raise FileNotFoundError(
        f"search_space_{c11_params.GRID}.json not found at {json_path}. "
        "Generate it from c12_15 before running the GA."
    )
with open(json_path, "r", encoding="utf-8") as f:
    optimizer_search_space = json.load(f)

print(f"Search space loaded: {len(optimizer_search_space)} variables")

stock_file = config.TIMBER_STOCK_PATH / f"complete_timber_{TRAINING_SCENARIO}.csv"
if not stock_file.exists():
    raise FileNotFoundError(f"Stock CSV not found: {stock_file}")

_df_input_stock = None
for _opts in [
    {"sep": ",",  "encoding": "utf-8"},
    {"sep": ";",  "encoding": "utf-8"},
    {"sep": ",",  "encoding": "latin1"},
    {"sep": ";",  "encoding": "latin1"},
]:
    try:
        _df = pd.read_csv(stock_file, **_opts)
        if _df.shape[1] > 1:
            _df_input_stock = _df
            print(f"Stock loaded: {len(_df)} elements  "
                  f"(sep='{_opts['sep']}', encoding='{_opts['encoding']}')")
            break
    except Exception:
        pass

if _df_input_stock is None:
    raise ValueError(f"Could not parse stock CSV at {stock_file}.")

df_input_stock = _df_input_stock
df_input_stock.columns = df_input_stock.columns.str.strip()

PREPARED_GNN_STOCK = stage_gnn.prepare_stock_for_gnn(df_input_stock)
print(f"GNN stock prepared: {len(PREPARED_GNN_STOCK)} elements")

MODEL_PREFIX = "ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0.863"

GA_CONFIG = {
    "fitness_weights": {
        "omega_1": 1.0,
        "omega_2": 1.0,
    },
    "new_stock_max_uses":    10,
    "min_reuse_fraction":    0.0,
    "penalty_fitness":       1e6,
    "use_one_time_bounds":   True,
    "bounds_probe_attempts": 40,
    "w_structural_start":    2.0,
    "w_structural_end":      0.8,
    "max_structural_infeas": 1.0,  # 1.0 = disabled
    "use_gnn":               True,
}

FIXED_NORMALIZATION_CONSTANTS = stage_fitness.get_default_normalization_constants()

_mrf = GA_CONFIG.get("min_reuse_fraction")
_mrf_str = f"{_mrf:.0%}" if _mrf else "None (unconstrained)"

print(f"\n{'='*55}")
print("GA SESSION READY")
print(f"{'='*55}")
print(f"  Search space variables:  {len(optimizer_search_space)}")
print(f"  Stock elements:          {len(df_input_stock)}")
print(f"  Model prefix:            {MODEL_PREFIX}")
print(f"  Fitness weights:         "
      f"ω1={GA_CONFIG['fitness_weights']['omega_1']}  "
      f"ω2={GA_CONFIG['fitness_weights']['omega_2']}")
print(f"  Structural schedule:     "
      f"ω4 {GA_CONFIG['w_structural_start']} → {GA_CONFIG['w_structural_end']}")
print(f"  Hard structural floor:   {GA_CONFIG['max_structural_infeas']:.0%}")
print(f"  New stock max uses:      {GA_CONFIG['new_stock_max_uses']}")
print(f"  Min reuse fraction:      {_mrf_str}")
print(f"  Fixed normalization:     {FIXED_NORMALIZATION_CONSTANTS}")
print(f"  Penalty fitness:         {GA_CONFIG['penalty_fitness']:.0e}")
print(f"  GNN in fitness:          {'Yes' if GA_CONFIG.get('use_gnn', True) else 'No (omega_4 zeroed)'}")
print(f"  TESTING mode:            {TESTING}")
print(f"{'='*55}\n")

display(df_input_stock.head(3))

# 2. Evaluation pipeline
Pre-checks before the GA run. Verify geometry topology and solve for one-time normalisation constants. Both cells should complete without errors before starting section 3.

## 2.1 Geometry check
Generates one random structure and verifies the expected node and member count. Set `RUN_GEOMETRY_CHECK = False` to skip during production runs.

In [ ]:
RUN_GEOMETRY_CHECK = True

EXPECTED_NODES   = 39
EXPECTED_MEMBERS = 120

if RUN_GEOMETRY_CHECK:
    geometry_out = stage_geometry.run_random_geometry_stage(
        optimizer_search_space=optimizer_search_space,
        sample_id=0,
    )

    df_vertices_example          = geometry_out["df_vertices"]
    df_edges_example             = geometry_out["df_edges"]
    df_geometry_overview_example = geometry_out["df_geometry_overview"]

    n_nodes   = len(df_vertices_example)
    n_members = len(df_edges_example)

    if n_nodes != EXPECTED_NODES:
        print(f"  WARNING: expected {EXPECTED_NODES} nodes, got {n_nodes}")
    else:
        print(f"  Nodes:   {n_nodes} / {EXPECTED_NODES} ✓")

    if n_members != EXPECTED_MEMBERS:
        print(f"  WARNING: expected {EXPECTED_MEMBERS} members, got {n_members}")
    else:
        print(f"  Members: {n_members} / {EXPECTED_MEMBERS} ✓")

    display(df_geometry_overview_example[["edge_id", "V1", "V2", "length_m"]].head(5))

else:
    print("Geometry check skipped.")

## 2.2 Bounds solve + smoke test
Probes the full pipeline to compute one-time normalisation constants, then runs a single end-to-end evaluation (geometry → feasibility → MILP → GNN → fitness) to confirm all stages are wired correctly before launching the GA.

In [ ]:
FIXED_NORMALIZATION_CONSTANTS, BOUNDS_SOURCE_INFO = (
    ga_eval._compute_one_time_normalization_constants(
        search_space = optimizer_search_space,
        df_stock     = df_input_stock,
        config_dict  = GA_CONFIG,
    )
)
print(f"Normalisation constants: {FIXED_NORMALIZATION_CONSTANTS}")
print(f"Source: {BOUNDS_SOURCE_INFO}")

print("\nRunning evaluator smoke test...")
_test_design = stage_geometry.sample_random_design(optimizer_search_space)
_test_eval   = ga_eval.evaluate_design_candidate(
    design_params        = _test_design,
    df_stock             = df_input_stock,
    fixed_norm_constants = FIXED_NORMALIZATION_CONSTANTS,
    config_dict          = GA_CONFIG,
    model_prefix         = MODEL_PREFIX,
    generation           = 0,
    max_generations      = 1,
    sample_id            = 99_999,
    verbose              = True,
    prepared_gnn_stock   = PREPARED_GNN_STOCK,
)
print(f"Smoke test status: {_test_eval['status']}")
if _test_eval["reason"] is not None:
    print(f"Reason: {_test_eval['reason']}")
print("Evaluator ready.")

# 3. GA run
Loads the surrogate bundle, wires the evaluation function, and runs CMA-ES. Tune `CMAESConfig` parameters (`popsize`, `n_generations`, `sigma_init`) to adjust exploration budget.

In [ ]:
import time
import c23_ga_algorithm as ga_algo
from c21_surrogate_io import load_surrogate_bundle

importlib.reload(ga_algo)
importlib.reload(ga_eval)

SURROGATE_BUNDLE = load_surrogate_bundle(prefix_sm=MODEL_PREFIX)
print(f"Bundle pre-loaded: {MODEL_PREFIX}")

evaluate_fn = ga_algo.make_evaluate_fn(
    evaluate_fn_raw      = ga_eval.evaluate_design_candidate,
    df_stock             = df_input_stock,
    fixed_norm_constants = FIXED_NORMALIZATION_CONSTANTS,
    config_dict          = GA_CONFIG,
    bundle               = SURROGATE_BUNDLE,
    prepared_gnn_stock   = PREPARED_GNN_STOCK,
    verbose              = True,
)

_base_fn = evaluate_fn
def evaluate_fn(params, generation=0, max_generations=1):
    print()
    t0 = time.time()
    fitness, res = _base_fn(params, generation, max_generations)
    elapsed = time.time() - t0
    if res:
        status  = res.get("status",        "?")
        milp    = res.get("milp_status",    "?")
        gnn     = res.get("gnn_feasibility")
        cost    = res.get("total_cost",     float("nan"))
        reuse   = res.get("reuse_fraction", float("nan"))
        w4      = res.get("w_structural",   float("nan"))
        gnn_str = f"{gnn:.3f}" if gnn is not None else "skip"
        print(
            f"  → {elapsed:.1f}s | {status} | MILP={milp} | "
            f"GNN={gnn_str} | cost={cost:.1f} | reuse={reuse:.3f} | "
            f"ω4={w4:.2f} | fit={fitness:.4f}"
        )
    else:
        print(f"  → {elapsed:.1f}s | fitness={fitness:.4f}  (no result dict)")
    return fitness, res


def _ss_to_bounds(ss):
    out = {}
    for k, v in ss.items():
        if v["type"] == "discrete":
            out[k] = (float(min(v["options"])), float(max(v["options"])))
        else:
            out[k] = (float(v["min"]), float(v["max"]))
    return out

es_search_space = _ss_to_bounds(optimizer_search_space)

es = ga_algo.CMAEvolutionStrategy(
    search_space = es_search_space,
    evaluate_fn  = evaluate_fn,
    config       = ga_algo.CMAESConfig(
        popsize          = 30,
        n_generations    = 250,
        sigma_init       = 0.25,
        sigma_min        = 1e-8,
        stagnation_limit = 30,
        log_every        = 5,
    ),
    seed = 42,
)

result = es.run()

# 4. Analysis
Inspect GA convergence, fitness trajectory, and the quality of the best-found solutions across the run.

In [ ]:
from workflows import c23_ga_analysis_export as ga_ae
importlib.reload(ga_ae)

analysis_out = ga_ae.run_analysis(
    result                 = result,
    fixed_norm_constants   = FIXED_NORMALIZATION_CONSTANTS,
    optimizer_search_space = optimizer_search_space,
    stagnation_limit       = es.config.stagnation_limit,
)

# 5. Export
Save optimised design candidates, run configuration, and GA metadata to disk for downstream use in Grasshopper and reporting.

In [ ]:
importlib.reload(ga_ae)

seed = (SEED, 0)

ga_config_export = dict(GA_CONFIG)
ga_config_export["seed"] = SEED
ga_config_export["seed_reference"] = seed

export_out = ga_ae.run_export(
    analysis_out         = analysis_out,
    result               = result,
    ga_config            = ga_config_export,
    fixed_norm_constants = FIXED_NORMALIZATION_CONSTANTS,
    model_prefix         = MODEL_PREFIX,
    bounds_source_info   = BOUNDS_SOURCE_INFO,
    es                   = es,
    df_stock             = df_input_stock,
    stock_source_path    = stock_file,
)
print("Saved to:", export_out["export_dir"])